# YOLOv8 Digit Strip Finder Training (Google Colab)

This notebook trains a YOLOv8 detector for finding the digit strip.

Expected dataset inputs:
- `ROI_640/` containing 640x640 images
- `ROI_640_labels/` containing YOLO `.txt` labels with the same stem names

Important:
- This model is a finder, not a reader.
- If you train on `ROI_640`, you should validate on `ROI_640`-like images too.
- Testing it on much larger raw originals is a distribution mismatch.

## 1. Runtime Setup

In Colab:
1. `Runtime > Change runtime type`
2. Set `Hardware accelerator` to `GPU`
3. Run the cells below

In [ ]:
!nvidia-smi || true
!pip install -q ultralytics==8.3.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths

Edit these paths to match your Google Drive.

In [ ]:
from pathlib import Path

# EDIT THESE
IMAGES_DIR = Path('/content/drive/MyDrive/DigitExtractor/ROI_640')
LABELS_DIR = Path('/content/drive/MyDrive/DigitExtractor/ROI_640_labels')
OUTPUT_DIR = Path('/content/drive/MyDrive/DigitExtractor/trained_yolo_models_colab')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('IMAGES_DIR =', IMAGES_DIR)
print('LABELS_DIR =', LABELS_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 3. Build YOLO Dataset Split

In [ ]:
import random
import shutil

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}

all_images = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

paired = []
missing_labels = []
for image_path in all_images:
    label_path = LABELS_DIR / f'{image_path.stem}.txt'
    if label_path.exists():
        paired.append((image_path, label_path))
    else:
        missing_labels.append(image_path.name)

print('paired:', len(paired))
print('missing labels:', len(missing_labels))
if missing_labels[:10]:
    print('sample missing labels:', missing_labels[:10])

assert len(paired) >= 2, 'Need at least 2 paired images/labels.'

random.seed(42)
random.shuffle(paired)
split_idx = max(1, int(len(paired) * 0.8))
if split_idx >= len(paired):
    split_idx = len(paired) - 1

train_pairs = paired[:split_idx]
val_pairs = paired[split_idx:]

dataset_root = Path('/content/yolo_digit_strip_dataset')
if dataset_root.exists():
    shutil.rmtree(dataset_root)

for sub in [
    'images/train', 'images/val', 'labels/train', 'labels/val'
]:
    (dataset_root / sub).mkdir(parents=True, exist_ok=True)

def copy_pairs(pairs, image_subdir, label_subdir):
    for image_path, label_path in pairs:
        shutil.copy2(image_path, dataset_root / image_subdir / image_path.name)
        shutil.copy2(label_path, dataset_root / label_subdir / label_path.name)

copy_pairs(train_pairs, 'images/train', 'labels/train')
copy_pairs(val_pairs, 'images/val', 'labels/val')

yaml_path = dataset_root / 'dataset.yaml'
yaml_path.write_text(
    '\n'.join([
        f'path: {dataset_root.as_posix()}',
        'train: images/train',
        'val: images/val',
        'names:',
        '  0: digit_strip',
        ''
    ]),
    encoding='utf-8'
)

print('train pairs:', len(train_pairs))
print('val pairs:', len(val_pairs))
print('dataset yaml:', yaml_path)

## 4. Quick Visual Sanity Check

This helps verify that the labels are aligned before training.

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample_image_path, sample_label_path = train_pairs[0]
img = cv2.imread(str(sample_image_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

for line in sample_label_path.read_text(encoding='utf-8').splitlines():
    parts = line.strip().split()
    if len(parts) != 5:
        continue
    _, cx, cy, bw, bh = map(float, parts)
    x1 = int((cx - bw / 2.0) * w)
    y1 = int((cy - bh / 2.0) * h)
    x2 = int((cx + bw / 2.0) * w)
    y2 = int((cy + bh / 2.0) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. Train YOLOv8

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
IMAGE_SIZE = 640
BATCH = 16

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    project=str(OUTPUT_DIR),
    name='yolov8_digit_strip',
    exist_ok=True
)

print('save_dir =', results.save_dir)

## 6. Evaluate with the Best `.pt` Model First

Use `.pt` first before worrying about `.onnx` or `.tflite`.

In [ ]:
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print(best_pt)
assert best_pt.exists(), 'best.pt was not created.'

In [ ]:
sample_val_image = val_pairs[0][0]
pred_model = YOLO(str(best_pt))
pred_results = pred_model.predict(source=str(sample_val_image), imgsz=640, conf=0.10)
pred_results[0].show()

## 7. Optional Export

Export after you confirm `.pt` works.

In [ ]:
exported_onnx = pred_model.export(format='onnx', imgsz=640)
print('onnx =', exported_onnx)

try:
    exported_tflite = pred_model.export(format='tflite', imgsz=640)
    print('tflite =', exported_tflite)
except Exception as exc:
    print('tflite export failed:', exc)

## 8. Download or Copy Results

The weights are already saved to your Google Drive output folder.